# YouTube Shorts Bulk Analysis - Advanced
## Find parent long-form videos using ytInitialData extraction
1. Extract shorts from dataset
2. Fetch YouTube HTML and extract ytInitialData JSON
3. Find watchEndpoint source video IDs
4. Cross-reference with description links for verification

In [133]:
# ⚙️  CONFIGURATION - TOGGLE HERE
USE_SAMPLE = True  # Set to True to test with sample, False to process all shorts
SAMPLE_SIZE = 20 # Number of shorts to process if USE_SAMPLE=True

# 📅 DATE FILTER - Only analyze shorts from this year onwards
START_YEAR = 2025  # Only process shorts from 2025 onwards
END_YEAR = 2026    # Process up to this year (2026)

print(f"Configuration:")
print(f"  USE_SAMPLE: {USE_SAMPLE}")
print(f"  SAMPLE_SIZE: {SAMPLE_SIZE}")
print(f"  DATE RANGE: {START_YEAR}-01-01 to {END_YEAR}-12-31")
print(f"\nTo change:")
print(f"  - Set USE_SAMPLE=False for full dataset analysis")
print(f"  - Adjust START_YEAR/END_YEAR to change date range")

Configuration:
  USE_SAMPLE: True
  SAMPLE_SIZE: 20
  DATE RANGE: 2025-01-01 to 2026-12-31

To change:
  - Set USE_SAMPLE=False for full dataset analysis
  - Adjust START_YEAR/END_YEAR to change date range


In [134]:
import pandas as pd
import numpy as np
import json
import random
import re
import subprocess
import requests
from pathlib import Path
from datetime import datetime
import time

# Playwright for rendering the Shorts overlay (JS-hydrated DOM)
from playwright.sync_api import sync_playwright

print("Libraries loaded successfully")
print(f"Started at: {datetime.now()}")

Libraries loaded successfully
Started at: 2026-05-30 15:56:06.207633


In [135]:
# Load dataset
youtube_data_path = Path('../Data/YouTube/videos_metadata.csv')

if youtube_data_path.exists():
    df = pd.read_csv(youtube_data_path, low_memory=False)
    print(f"✅ Dataset loaded!")
    print(f"Total videos: {len(df)}")
    
    # Parse publishedAt as datetime and extract year
    if 'publishedAt' in df.columns:
        df['publishedAt'] = pd.to_datetime(df['publishedAt'], errors='coerce')
        df['year'] = df['publishedAt'].dt.year
        print(f"Date range in dataset: {df['publishedAt'].min()} to {df['publishedAt'].max()}")
    
    # Filter shorts
    if 'is_short' in df.columns:
        all_shorts = df[df['is_short'] == True].copy()
        all_longs = df[df['is_short'] == False].copy()
        
        print(f"\nTotal shorts: {len(all_shorts)}")
        print(f"Total longs: {len(all_longs)}")
        
        # Apply YEAR FILTER
        print(f"\n📅 Applying date filter: {START_YEAR} onwards")
        shorts_df = all_shorts[
            (all_shorts['year'] >= START_YEAR) & 
            (all_shorts['year'] <= END_YEAR)
        ].copy().reset_index(drop=True)
        
        print(f"Shorts after date filter: {len(shorts_df)} (from {len(all_shorts)} total)")
        
        if len(shorts_df) > 0:
            print(f"\nDate range of filtered shorts: {shorts_df['publishedAt'].min()} to {shorts_df['publishedAt'].max()}")
            print(f"\n📊 Year distribution:")
            year_counts = shorts_df['year'].value_counts().sort_index()
            for year, count in year_counts.items():
                print(f"  {int(year)}: {count} shorts")
        else:
            print(f"⚠️  No shorts found in date range {START_YEAR}-{END_YEAR}")
        
        # Apply STRATIFIED RANDOM SAMPLE by channel title
        if USE_SAMPLE and len(shorts_df) > SAMPLE_SIZE:
            print(f"\n⚠️  USING STRATIFIED RANDOM SAMPLE MODE (by channel)")
            print(f"Sampling {SAMPLE_SIZE} random shorts stratified by channel...\n")
            
            sampled_list = []
            total_shorts_count = len(shorts_df)
            channels = shorts_df['channelTitle'].unique()
            
            for channel in channels:
                channel_data = shorts_df[shorts_df['channelTitle'] == channel]
                proportion = len(channel_data) / total_shorts_count
                n_samples = max(1, int(SAMPLE_SIZE * proportion))
                
                if len(channel_data) > 0:
                    sampled = channel_data.sample(n=min(n_samples, len(channel_data)), random_state=42)
                    sampled_list.append(sampled)
            
            shorts_df = pd.concat(sampled_list, ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
            
            print(f"✅ Stratified random sample created: {len(shorts_df)} shorts")
            print(f"\n📊 Channel representation in sample:")
            for channel, count in shorts_df['channelTitle'].value_counts().items():
                print(f"  {channel}: {count} shorts")
    else:
        print("❌ No 'is_short' column")
        shorts_df = None
else:
    print(f"❌ Dataset not found")
    shorts_df = None

✅ Dataset loaded!
Total videos: 12839
Date range in dataset: 2008-02-14 17:04:02+00:00 to 2026-03-01 13:47:43+00:00

Total shorts: 4356
Total longs: 8483

📅 Applying date filter: 2025 onwards
Shorts after date filter: 1301 (from 4356 total)

Date range of filtered shorts: 2025-01-01 09:53:09+00:00 to 2026-03-01 13:47:43+00:00

📊 Year distribution:
  2025: 1056 shorts
  2026: 245 shorts

⚠️  USING STRATIFIED RANDOM SAMPLE MODE (by channel)
Sampling 20 random shorts stratified by channel...

✅ Stratified random sample created: 26 shorts

📊 Channel representation in sample:
  Daniele Doesn't Matter: 3 shorts
  Christian Manzoni: 2 shorts
  Alessandra Spadina: 2 shorts
  Camihawke: 1 shorts
  Adriano Moretti: 1 shorts
  Davide Zambelli: 1 shorts
  Il Matricomio: 1 shorts
  Bella Gianda Official: 1 shorts
  Gli Autogol: 1 shorts
  Smibie Channel: 1 shorts
  Cotto al Dente: 1 shorts
  Sara La Rusca: 1 shorts
  Raissa & Momo: 1 shorts
  The Lady: 1 shorts
  roccotnl: 1 shorts
  fitgreta: 1 sh

In [136]:
shorts_df['channelTitle'].value_counts().head(10)

channelTitle
Daniele Doesn't Matter    3
Christian Manzoni         2
Alessandra Spadina        2
Camihawke                 1
Adriano Moretti           1
Davide Zambelli           1
Il Matricomio             1
Bella Gianda Official     1
Gli Autogol               1
Smibie Channel            1
Name: count, dtype: int64

In [137]:
short_df = shorts_df[shorts_df['is_short'] == True].copy()

In [138]:
# =========================================================================
# PLAYWRIGHT-BASED EXTRACTION
# Renders the Short's JS overlay and reads the
# <a class="ytReelMultiFormatLinkViewModelEndpoint"> element directly.
# Records the connection whether it points to a /watch?v= (long) or
# a /shorts/ (short) video.
# =========================================================================

# Consent cookie to bypass YouTube's EU/Italy consent interstitial
# (consent.youtube.com). Without this the overlay never renders.
YT_CONSENT_COOKIE = {
    "name": "SOCS",
    "value": "CAISEwgDEgk0ODE3Nzk3MjQaAmVuIAEaBgiA_LyaBg",
    "domain": ".youtube.com",
    "path": "/",
}

MULTIFORMAT_SELECTOR = "a.ytReelMultiFormatLinkViewModelEndpoint"


def start_browser(playwright, headless=True):
    """Launch a Chromium browser + context with consent cookie pre-set."""
    browser = playwright.chromium.launch(headless=headless)
    context = browser.new_context(
        user_agent=("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                    "(KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36"),
        locale="en-US",
    )
    context.add_cookies([YT_CONSENT_COOKIE])
    return browser, context


def parse_link_target(href):
    """
    Given an overlay href, return (link_type, linked_video_id).
      link_type: 'long'  for /watch?v=...
                 'short' for /shorts/...
                 None    for anything else
    """
    if not href:
        return None, None
    m = re.search(r"/watch\?v=([a-zA-Z0-9_-]{11})", href)
    if m:
        return "long", m.group(1)
    m = re.search(r"/shorts/([a-zA-Z0-9_-]{11})", href)
    if m:
        return "short", m.group(1)
    return None, None


def get_multiformat_link(page, short_id, timeout_ms=15000):
    """
    Navigate to a Short and extract the multi-format link overlay element.

    Records the connection regardless of whether it links to a long video
    (/watch?v=) or another short (/shorts/).

    Returns a dict:
      {
        'href':  '/watch?v=...' or '/shorts/...' or None,
        'label': aria-label / title text,
        'link_type': 'long' | 'short' | None,
        'linked_video_id': the 11-char id of the linked video (long or short),
      }
    or None if the element never rendered.
    """
    url = f"https://www.youtube.com/shorts/{short_id}"
    try:
        page.goto(url, wait_until="domcontentloaded", timeout=30000)

        if "consent.youtube.com" in page.url:
            return None  # consent bypass failed

        page.wait_for_selector(MULTIFORMAT_SELECTOR, state="attached", timeout=timeout_ms)

        rows = page.eval_on_selector_all(
            MULTIFORMAT_SELECTOR,
            "els => els.map(e => ({href: e.getAttribute('href'), "
            "label: e.getAttribute('aria-label')}))"
        )

        # Prefer a /watch?v= (long video) link; otherwise the first link to a
        # *different* video than the short itself; otherwise the self-link.
        chosen = None
        for r in rows:
            lt, vid = parse_link_target(r.get("href"))
            if lt == "long":
                chosen = r
                break
        if chosen is None:
            for r in rows:
                lt, vid = parse_link_target(r.get("href"))
                if vid and vid != short_id:
                    chosen = r
                    break
        if chosen is None and rows:
            chosen = rows[0]
        if chosen is None:
            return None

        href = chosen.get("href")
        link_type, linked_video_id = parse_link_target(href)

        return {
            "href": href,
            "label": chosen.get("label"),
            "link_type": link_type,
            "linked_video_id": linked_video_id,
        }
    except Exception:
        return None


# Fallback: download JSON metadata via yt-dlp (for description links)
def download_video_json(video_id):
    """Download complete JSON metadata for a video using yt-dlp"""
    try:
        cmd = ['yt-dlp', '--dump-json', '--skip-download',
               f'https://www.youtube.com/watch?v={video_id}']
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        if result.returncode == 0:
            return json.loads(result.stdout)
    except Exception:
        pass
    return None


def extract_youtube_links_from_description(text, exclude_id=None):
    """Extract YouTube video IDs from description text (fallback method)"""
    if not text:
        return []
    links = []
    patterns = [
        r'(?:https?://)?(?:www\.)?youtube\.com/watch\?v=([a-zA-Z0-9_-]{11})',
        r'(?:https?://)?youtu\.be/([a-zA-Z0-9_-]{11})',
        r'(?:https?://)?(?:www\.)?youtube\.com/shorts/([a-zA-Z0-9_-]{11})'
    ]
    for pattern in patterns:
        for match in re.finditer(pattern, text):
            vid_id = match.group(1)
            if exclude_id is None or vid_id != exclude_id:
                links.append(vid_id)
    return list(dict.fromkeys(links))


print("✅ Helper functions defined - Playwright overlay extractor (long + short) + yt-dlp fallback")

✅ Helper functions defined - Playwright overlay extractor (long + short) + yt-dlp fallback


In [139]:
# Process all shorts with Playwright
# Playwright's SYNC API cannot run inside Jupyter on Windows: Jupyter forces a
# SelectorEventLoop (for Tornado), and that loop type cannot spawn the browser
# subprocess -> NotImplementedError. The bullet-proof fix is to run the
# extraction in its OWN process via shorts_extractor.py, which gets a proper
# main-thread ProactorEventLoop. We call it with sys.executable so it uses the
# exact same Python/conda env as this kernel.
import sys

print("\n" + "="*80)
print("PROCESSING SHORTS - PLAYWRIGHT OVERLAY EXTRACTION (subprocess)")
print("="*80)

if shorts_df is not None and len(shorts_df) > 0:
    output_dir = Path('./shorts_analysis')
    output_dir.mkdir(parents=True, exist_ok=True)

    extractor = Path('shorts_extractor.py').resolve()
    input_csv = output_dir / '_extractor_input.csv'
    output_csv = output_dir / '_extractor_output.csv'

    # Write the shorts to process (only the columns the extractor needs)
    cols_needed = ['videoId', 'title', 'channelTitle', 'publishedAt', 'year']
    shorts_df[cols_needed].to_csv(input_csv, index=False, encoding='utf-8')

    print(f"\nLaunching extractor on {len(shorts_df)} shorts...")
    print(f"  script : {extractor}")
    print(f"  python : {sys.executable}\n")

    # Run the extractor as a separate process and stream its output live
    proc = subprocess.Popen(
        [sys.executable, str(extractor), str(input_csv), str(output_csv)],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding='utf-8', bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='')
    proc.wait()

    if output_csv.exists():
        results_df = pd.read_csv(output_csv)
        print(f"\n✅ Processed {len(results_df)} shorts")
    else:
        print(f"\n❌ Extractor did not produce output (exit code {proc.returncode})")
        results_df = None
else:
    print(f"❌ No shorts data available")
    results_df = None


PROCESSING SHORTS - PLAYWRIGHT OVERLAY EXTRACTION (subprocess)

Launching extractor on 26 shorts...
  script : D:\Polythecninco di Milano\AFB_Lab\Ali\shorts_extractor.py
  python : d:\conda_envs\ma_env\python.exe

Processing 26 shorts...
[1/26] ZW0Linfanpk (2025) https://www.youtube.com/shorts/ZW0Linfanpk
    no overlay
[2/26] Y5j4lazJzzo (2025) https://www.youtube.com/shorts/Y5j4lazJzzo
    no overlay
[3/26] eNU5FcmEwTc (2026) https://www.youtube.com/shorts/eNU5FcmEwTc
    no overlay
[4/26] 3dUWv3HTS60 (2026) https://www.youtube.com/shorts/3dUWv3HTS60
    no overlay
[5/26] b4t6fdYh8ak (2025) https://www.youtube.com/shorts/b4t6fdYh8ak
    LONG -> gE2Phwx-nKg
[6/26] yJjNhBzE650 (2026) https://www.youtube.com/shorts/yJjNhBzE650
    no overlay
[7/26] tDNOWjlhzXw (2026) https://www.youtube.com/shorts/tDNOWjlhzXw
    no overlay
[8/26] o_-6ecCQsuY (2025) https://www.youtube.com/shorts/o_-6ecCQsuY
    no overlay
[9/26] 88weCh62SeU (2025) https://www.youtube.com/shorts/88weCh62SeU
    SHORT (

In [140]:
# Display the connection dataset
print("\n" + "="*80)
print("SHORT → VIDEO CONNECTION DATASET")
print("="*80)

if results_df is not None and len(results_df) > 0:
    # ---- 1) The captured overlay element + recorded connection for EVERY short ----
    cols = ['short_id', 'connection_type', 'connected_id', 'overlay_label', 'detection_method']
    conn_dataset = results_df[cols].copy()
    conn_dataset.columns = ['Short ID', 'Type', 'Connected ID', 'Overlay label', 'Method']

    print(f"\n🔗 CAPTURED CONNECTIONS (all {len(conn_dataset)} shorts):\n")
    with pd.option_context('display.max_colwidth', 40, 'display.width', 200):
        print(conn_dataset.to_string(index=False))

    # ---- 2) Breakdown by connection type ----
    print("\n" + "-"*80)
    print("\n📊 CONNECTION TYPE BREAKDOWN:")
    type_counts = results_df['connection_type'].value_counts()
    for t, c in type_counts.items():
        print(f"  {t:>6}: {c} ({c/len(results_df)*100:.1f}%)")

    # ---- 3) All recorded connections with clickable links ----
    connected = results_df[results_df['status'] == 'connected'].copy()
    print("\n" + "-"*80)
    if len(connected) > 0:
        print(f"\n✅ {len(connected)} SHORTS WITH A RECORDED CONNECTION\n")
        for idx, (_, row) in enumerate(connected.iterrows(), 1):
            emoji = "📹" if row['connection_type'] == 'long' else "📱"
            print(f"{idx}. SHORT : {row['short_id']}")
            print(f"   📱 {row['short_url']}")
            print(f"   ↓ {row['connection_type'].upper()} link ({row['detection_method']})")
            print(f"   {emoji} {row['connected_id']}")
            print(f"   🔗 {row['connected_url']}")
            print()
    else:
        print("\n❌ No connections recorded in this sample.")
else:
    print("\n❌ No results. Run the processing cell first.")


SHORT → VIDEO CONNECTION DATASET

🔗 CAPTURED CONNECTIONS (all 26 shorts):

   Short ID  Type Connected ID                                                                                      Overlay label       Method
ZW0Linfanpk  none          NaN                                                                                                NaN    not_found
Y5j4lazJzzo  none          NaN                                                                                                NaN    not_found
eNU5FcmEwTc  none          NaN                                                                                                NaN    not_found
3dUWv3HTS60  none          NaN                                                                                                NaN    not_found
b4t6fdYh8ak  long  gE2Phwx-nKg                                  IL CALENDARIO BEAUTY PIU COSTOSO‼️💸 CULT BEAUTY NE VALE LA PENA?! overlay_link
yJjNhBzE650  none          NaN                                    

In [ ]:
# View JSON dataset
print("\n" + "="*80)
print("JSON METADATA DATASET")
print("="*80)

json_dir = Path('./shorts_analysis/json_data')
if json_dir.exists():
    json_files = list(json_dir.glob('*.json'))
    print(f"\n✅ Found {len(json_files)} JSON files in {json_dir}")
    
    if len(json_files) > 0:
        # Load and display key fields from JSON
        json_data_list = []
        
        for json_file in json_files[:10]:  # Show first 10
            try:
                with open(json_file, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    json_data_list.append({
                        'video_id': data.get('id', 'N/A'),
                        'title': data.get('title', 'N/A')[:50],  # Truncate long titles
                        'duration': data.get('duration', 'N/A'),
                        'description_length': len(data.get('description', '')),
                        'upload_date': data.get('upload_date', 'N/A'),
                        'view_count': data.get('view_count', 'N/A'),
                    })
            except Exception as e:
                pass
        
        if json_data_list:
            json_df = pd.DataFrame(json_data_list)
            print("\n📊 SAMPLE OF JSON DATA (first 10 videos):\n")
            print(json_df.to_string(index=False))
            
            print(f"\n\n📥 FULL JSON SAMPLES:")
            print("-" * 80)
            for idx, json_file in enumerate(json_files[:3], 1):
                try:
                    with open(json_file, 'r', encoding='utf-8') as f:
                        data = json.load(f)
                        print(f"\n{idx}. {data.get('id', 'N/A')} - {data.get('title', 'N/A')[:60]}")
                        print(f"   Description: {data.get('description', 'N/A')[:100]}...")
                        print(f"   Duration: {data.get('duration', 'N/A')}s")
                        print(f"   Views: {data.get('view_count', 'N/A')}")
                except:
                    pass
else:
    print(f"\n❌ No JSON data directory found. Run processing cell first!")

In [ ]:
# Export results to CSV
if results_df is not None:
    output_dir = Path('./shorts_analysis')
    output_dir.mkdir(exist_ok=True)

    # Add date range and mode to filename
    date_suffix = f"_{START_YEAR}-{END_YEAR}"
    mode_suffix = '_sample' if USE_SAMPLE else '_full'
    suffix = date_suffix + mode_suffix

    # Full results
    csv_path = output_dir / f'shorts_connections{suffix}.csv'
    results_df.to_csv(csv_path, index=False)
    print(f"\n✅ Full results exported to: {csv_path}")

    # Only shorts with a recorded connection
    connected_df = results_df[results_df['status'] == 'connected']
    if len(connected_df) > 0:
        conn_csv_path = output_dir / f'shorts_connection_mapping{suffix}.csv'
        connected_df.to_csv(conn_csv_path, index=False)
        print(f"✅ Connection mappings exported to: {conn_csv_path}")

    # Summary stats
    stats_path = output_dir / f'summary_stats{suffix}.json'
    type_counts = results_df['connection_type'].value_counts().to_dict()
    stats = {
        'date_range': f"{START_YEAR}-{END_YEAR}",
        'mode': 'sample' if USE_SAMPLE else 'full',
        'sample_size': SAMPLE_SIZE if USE_SAMPLE else None,
        'total_shorts': len(results_df),
        'shorts_connected': len(connected_df),
        'shorts_no_connection': len(results_df[results_df['status'] == 'no_connection']),
        'percentage_connected': round(len(connected_df)/len(results_df)*100, 2) if len(results_df) > 0 else 0,
        'connection_type_counts': {str(k): int(v) for k, v in type_counts.items()},
        'found_via_overlay': int((results_df['detection_method'] == 'overlay_link').sum()),
        'found_via_description': int((results_df['detection_method'] == 'description').sum()),
        'unique_connected_videos': connected_df['connected_id'].nunique() if len(connected_df) > 0 else 0,
        'analysis_date': datetime.now().isoformat()
    }

    with open(stats_path, 'w') as f:
        json.dump(stats, f, indent=2)

    print(f"✅ Stats exported to: {stats_path}")

In [ ]:
# Final Summary Table
print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)

if results_df is not None:
    print(f"\n📈 SUMMARY TABLE:")
    print("-" * 80)

    n_long = int((results_df['connection_type'] == 'long').sum())
    n_short = int((results_df['connection_type'] == 'short').sum())
    n_none = int((results_df['connection_type'] == 'none').sum())

    summary_data = {
        'Category': [
            'Total Shorts',
            'Connected (any)',
            '  → to LONG video',
            '  → to SHORT video',
            'No connection',
            'Found via overlay',
            'Found via description',
        ],
        'Count': [
            len(results_df),
            len(results_df[results_df['status'] == 'connected']),
            n_long,
            n_short,
            n_none,
            int((results_df['detection_method'] == 'overlay_link').sum()),
            int((results_df['detection_method'] == 'description').sum()),
        ]
    }
    summary_table = pd.DataFrame(summary_data)
    summary_table['Percentage'] = (summary_table['Count'] / len(results_df) * 100).round(1).astype(str) + '%'
    print(summary_table.to_string(index=False))

    print(f"\n📁 All results saved to: ./shorts_analysis/")
    print(f"   - shorts_connections{suffix}.csv (all results)")
    if len(results_df[results_df['status'] == 'connected']) > 0:
        print(f"   - shorts_connection_mapping{suffix}.csv (recorded connections)")
    print(f"   - summary_stats{suffix}.json (statistics)")

    print(f"\n⏰ Date range analyzed: {START_YEAR} - {END_YEAR}")
    if USE_SAMPLE:
        print(f"⚠️  SAMPLE MODE: Processing {len(results_df)} shorts (stratified by channel)")
        print(f"   To analyze FULL dataset: Change USE_SAMPLE = False in first cell")